# 09 · Fehlerbehandlung

**Lernziele**

- Du liest einen Traceback und findest die Fehlerursache.
- Du fängst Fehler mit `try` / `except` / `else` / `finally` ab.
- Du löst selbst Fehler mit `raise` aus.
- Du kennst eigene Fehlerklassen und das `with`-Statement.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 2 und 9.

## Tracebacks lesen

Fehler sind normal. Wichtig ist, die Meldung zu lesen, und zwar **von unten nach oben**:

1. Letzte Zeile: Fehlertyp und Beschreibung
2. Darüber: die Zeile, in der es passiert ist
3. Noch weiter oben: der Weg dorthin (welche Funktion hat welche aufgerufen)

Lies den Traceback zuerst selbst. Eine Chat-KI bittest du höchstens darum, dir eine Meldung zu erklären, die du nicht verstehst.

In [ ]:
def average(numbers):
    return sum(numbers) / len(numbers)


average([])

So liest du den Traceback der Zelle oben:

![Traceback von average([]) mit drei Markierungen: zuerst die letzte Zeile mit ZeroDivisionError lesen, dann Zeile 2 in average, wo der Fehler passiert ist, dann Zeile 5, die average aufgerufen hat.](../assets/09_traceback_lesen.png)

Häufige Fehlertypen:

| Fehler | Ursache | Beispiel |
| --- | --- | --- |
| `SyntaxError` | Code ist kein gültiges Python | fehlender Doppelpunkt |
| `NameError` | Name unbekannt | Tippfehler, Zelle nicht ausgeführt |
| `TypeError` | falscher Typ | `"a" + 1` |
| `ValueError` | richtiger Typ, falscher Wert | `int("drei")` |
| `IndexError` | Index außerhalb der Liste | `[1, 2][5]` |
| `KeyError` | Schlüssel fehlt im Dictionary | `{}["x"]` |
| `ZeroDivisionError` | Division durch 0 | `1 / 0` |
| `FileNotFoundError` | Datei existiert nicht | `open("gibtsnicht.txt")` |

**Kurzübung 1 (Traceback lesen):** Führe die Zelle unten aus und lies den Traceback von unten nach oben. Welcher Fehlertyp ist aufgetreten? In welcher Zeile und in welcher Funktion? Welche Funktion hat diese aufgerufen, und welcher Eintrag der Bestellung ist schuld?

In [ ]:
prices = {"Kaffee": 4.20, "Tee": 3.10}


def line_total(product, quantity):
    return prices[product] * quantity


def order_total(order):
    total = 0
    for product, quantity in order:
        total += line_total(product, quantity)
    return total


order_total([("Kaffee", 2), ("Kakao", 1), ("Tee", 3)])

In [ ]:
# Letzte Zeile: KeyError: 'Kakao', der Schlüssel "Kakao" fehlt in prices.
# Darüber: return prices[product] * quantity in der Funktion line_total.
# Weiter oben: order_total hat line_total in ihrer Schleife aufgerufen.
# Schuld ist der Eintrag ("Kakao", 1): Für Kakao ist kein Preis hinterlegt.

## Fehler abfangen mit `try` / `except`

Code, der fehlschlagen kann, steht in `try`. Im passenden `except` reagierst du darauf.

In [ ]:
user_input = "drei"
try:
    number = int(user_input)
    print(f"Das Doppelte ist {number * 2}")
except ValueError:
    print(f"'{user_input}' ist keine Zahl")

Fange immer **konkrete** Fehlertypen ab. Ein nacktes `except:` oder `except Exception:` verschluckt auch Fehler, die du sehen willst, etwa einen Tippfehler im Variablennamen (`NameError`). *Errors should never pass silently.*

Mit `as` bekommst du das Fehlerobjekt:

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError as error:
    print(f"Rechenfehler: {error}")

**Kurzübung 2 (Die Meldung stimmt nicht):** Der folgende Code meldet immer „Bitte eine ganze Zahl eingeben“, obwohl `quantity = "3"` gültig ist. Finde den Fehler und behebe ihn.

- Fange nur den Fehler ab, den du erwartest, und beobachte, was dann passiert.

In [ ]:
price = 4.20
quantity = "3"
try:
    total = int(quantity) * prise
    print(f"Summe: {total:.2f} €")
except:
    print("Bitte eine ganze Zahl eingeben")

In [ ]:
price = 4.20
quantity = "3"
try:
    total = int(quantity) * price  # vorher: prise -> NameError, vom nackten except verschluckt
    print(f"Summe: {total:.2f} €")
except ValueError:
    print("Bitte eine ganze Zahl eingeben")
# Mit "except ValueError" und dem Tippfehler bricht die Zelle mit
# NameError: name 'prise' is not defined ab, statt den Fehler zu verschlucken.

## Mehrere Fehlertypen

Für verschiedene Fehler schreibst du mehrere `except`-Blöcke. Sollen mehrere Fehler gleich behandelt werden, zählst du sie in einem `except` auf. Seit Python 3.14 brauchst du dafür keine Klammern mehr, solange kein `as` folgt. Älterer Code (vor Python 3.14) schreibt `except (ValueError, TypeError):` mit Klammern; beides bedeutet dasselbe.

In [ ]:
def to_number(value):
    try:
        return int(value)
    except ValueError, TypeError:  # "drei" ist ein ValueError, None ein TypeError
        return None


print(to_number("42"), to_number("drei"), to_number(None))

**Kurzübung 3 (Stückpreis berechnen):** Eine Rechnung über 12,60 € soll auf die gelieferte Stückzahl umgelegt werden; die Stückzahl kommt als Text. Gib für jeden Eintrag in `quantities` den Stückpreis oder eine passende Meldung aus, mit einem eigenen `except` je Fehlertyp. Erwartet für `"4"`: 3.15 €.

In [ ]:
invoice_total = 12.60
quantities = ["4", "zwei", "0"]

In [ ]:
for text in quantities:
    try:
        print(f"Stückpreis: {invoice_total / int(text):.2f} €")
    except ValueError:
        print(f"'{text}' ist keine ganze Zahl")
    except ZeroDivisionError:
        print("Menge 0: kein Stückpreis möglich")

## Fehlermeldungen helfen mit

Aktuelle Python-Versionen schlagen bei Tippfehlern vor, was du gemeint haben könntest, etwa `Did you mean '.append'?`. Wer aus Java oder JavaScript kommt, schreibt gern `push` statt `append`. Lies deshalb immer die letzte Zeile des Tracebacks bis zum Ende. Jupyter zeigt den Vorschlag allerdings nicht an:

In [ ]:
numbers = [1, 2]
numbers.push(3)

Im Terminal erscheint er. Mit `!` startest du ein Programm wie in Notebook 08; `python -c` führt eine einzelne Codezeile aus. Achte auf das Ende der letzten Zeile:

In [ ]:
!uv run python -c "[1, 2].push(3)"

**Kurzübung 4 (Drei Tippfehler):** Die Zelle unten enthält drei Tippfehler. Führe sie aus, lies die letzte Zeile des Tracebacks, behebe den gemeldeten Fehler und führe sie erneut aus, bis sie `3 Teilnehmende` und `ADA, LINUS, GRACE` ausgibt.

In [ ]:
attendees = ["Ada", "Linus"]
attendees.apend("Grace")
print(f"{lenn(attendees)} Teilnehmende")
print(", ".join(attendees).uper())

In [ ]:
attendees = ["Ada", "Linus"]
attendees.append("Grace")  # vorher apend: AttributeError
print(f"{len(attendees)} Teilnehmende")  # vorher lenn: NameError
print(", ".join(attendees).upper())  # vorher uper: AttributeError
# Python meldet immer nur den ersten Fehler; nach jeder Korrektur kommt der nächste.

## `else` und `finally`

- `else` läuft nur, wenn **kein** Fehler aufgetreten ist.
- `finally` läuft **immer**, z. B. zum Aufräumen.

![Ablauf von parse_number: Nach try führt ein ValueError (z. B. bei "abc") in den except-Block, ohne Fehler (z. B. bei "3.5") in den else-Block; beide Wege laufen danach durch finally, bevor die Funktion zum Aufrufer zurückkehrt.](../assets/09_try_except_ablauf.png)

In [ ]:
def parse_number(text):
    try:
        number = float(text)
    except ValueError:
        print(f"'{text}' ist ungültig")
        return None
    else:
        print(f"'{text}' gelesen")
        return number
    finally:
        print("Prüfung beendet")


parse_number("3.5")
parse_number("abc")

**Kurzübung 5 (Reihenfolge vorhersagen):** Sage vorher, welche Zeilen die folgende Zelle in welcher Reihenfolge ausgibt, und begründe. Führe sie danach aus.

In [ ]:
print(parse_number("1,5"))

In [ ]:
# '1,5' ist ungültig
# Prüfung beendet
# None
# float("1,5") scheitert am Komma, also läuft except und gibt None zurück.
# finally läuft trotzdem noch, bevor die Funktion verlassen wird.
# Erst danach gibt print den Rückgabewert None aus.

## Eingaben robust machen

Ein typisches Muster: so lange fragen, bis die Eingabe passt.

In [ ]:
while True:
    try:
        age = int(input("Alter: "))
        break
    except ValueError:
        print("Bitte eine ganze Zahl eingeben.")
print(f"Alter: {age}")

**Kurzübung 6 (Plausibles Alter):** Ändere die Schleife oben so, dass sie auch bei einer Zahl außerhalb von 0 bis 120 erneut fragt, mit der Meldung „Bitte ein Alter zwischen 0 und 120 eingeben.“

In [ ]:
while True:
    try:
        age = int(input("Alter: "))
    except ValueError:
        print("Bitte eine ganze Zahl eingeben.")
        continue
    if 0 <= age <= 120:
        break
    print("Bitte ein Alter zwischen 0 und 120 eingeben.")
print(f"Alter: {age}")

## Fehler selbst auslösen mit `raise`

Bekommt deine Funktion ungültige Werte, meldet sie das mit `raise`. Der Aufrufer entscheidet, wie er damit umgeht.

`None` zurückgeben passt, wenn „kein Ergebnis“ ein normaler Fall ist, etwa eine Suche ohne Treffer. `raise` passt, wenn der Aufruf so nicht sinnvoll ist und der Aufrufer es nicht übersehen darf.

In [ ]:
def withdraw(balance, amount):
    if amount <= 0:
        raise ValueError(f"Betrag muss positiv sein, war {amount}")
    if amount > balance:
        raise ValueError("Kontostand reicht nicht aus")
    return balance - amount


print(withdraw(100, 30))
print(withdraw(100, 300))

**Kurzübung 7 (Tageslimit):** Ergänze `withdraw` um eine Prüfung: Mehr als 1000 € auf einmal lösen einen `ValueError` mit der Meldung „Tageslimit von 1000 € überschritten“ aus. Rufe `withdraw(5000, 1500)` in `try` / `except ValueError as error` auf und gib die Meldung aus.

In [ ]:
def withdraw(balance, amount):
    if amount <= 0:
        raise ValueError(f"Betrag muss positiv sein, war {amount}")
    if amount > 1000:
        raise ValueError("Tageslimit von 1000 € überschritten")
    if amount > balance:
        raise ValueError("Kontostand reicht nicht aus")
    return balance - amount


try:
    print(withdraw(5000, 1500))
except ValueError as error:
    print(f"Abgelehnt: {error}")

## Eigene Fehlerklassen

Für eigene Fehlerarten erbst du von einer passenden Fehlerklasse. Mehr ist oft nicht nötig: Der Docstring reicht als Inhalt der Klasse; ohne Docstring schreibst du `pass`.

In [ ]:
class InsufficientFundsError(ValueError):
    """Wird ausgelöst, wenn das Konto nicht gedeckt ist."""


def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(f"{amount} € angefordert, nur {balance} € verfügbar")
    return balance - amount


try:
    withdraw(100, 300)
except InsufficientFundsError as error:
    print(f"Abgelehnt: {error}")

> Wie Klassen und Vererbung im Detail funktionieren, lernst du in Notebook 12.

**Kurzübung 8 (Welches except greift?):** Sage vorher, welche Ausgabe die folgende Zelle erzeugt, und begründe. Führe sie danach aus.

In [ ]:
try:
    withdraw(100, 300)
except ValueError as error:
    print(f"ValueError: {error}")

In [ ]:
# Ausgabe: ValueError: 300 € angefordert, nur 100 € verfügbar
# withdraw löst InsufficientFundsError aus. Diese Klasse erbt von ValueError,
# und except ValueError fängt auch alle Unterklassen. Darum erbt man von einer
# passenden Fehlerklasse: Code, der nur ValueError kennt, funktioniert weiter.

## Ressourcen sicher schließen mit `with`

Dateien, Netzwerkverbindungen usw. müssen geschlossen werden, auch wenn ein Fehler auftritt. `with` erledigt das automatisch. Mehr dazu in Notebook 10.

In [ ]:
try:
    with open("../daten/numbers.txt", encoding="utf-8") as file:
        count = int(file.readline())
        value = float(file.readline())
    print(count, value)
except FileNotFoundError:
    print("Datei nicht gefunden")
except ValueError:
    print("Keine gültige Zahl in Zeile 1 oder 2")

**Kurzübung 9 (Datei fehlt):** Kopiere die Zelle oben und ändere in der Kopie den Dateinamen in `../daten/zahlen.txt`. Welche Meldung erscheint?

- **Untersuchen:** Was passiert, wenn du zusätzlich den Block `except FileNotFoundError` entfernst?

In [ ]:
try:
    with open("../daten/zahlen.txt", encoding="utf-8") as file:
        count = int(file.readline())
        value = float(file.readline())
    print(count, value)
except FileNotFoundError:
    print("Datei nicht gefunden")
except ValueError:
    print("Keine gültige Zahl in Zeile 1 oder 2")
# Ausgabe: Datei nicht gefunden
# Ohne "except FileNotFoundError" bricht die Zelle mit FileNotFoundError ab:
# except ValueError passt nicht zu diesem Fehlertyp.

## Zusammenfassung

- Tracebacks von unten nach oben lesen.
- `try` / `except Fehlertyp` fängt konkrete Fehler ab, `as error` liefert Details.
- `else` läuft ohne Fehler, `finally` immer.
- `raise ValueError("…")` meldet ungültige Werte.
- Eigene Fehler: `class MyError(ValueError): ...`
- `with` schließt Ressourcen automatisch.